# Notebook 07 — Merge Everything

This notebook merges the cleaned monthly weather, soil moisture profiles, light & temperature extremes, and monthly crop production datasets into a single cohesive, district-monthly master dataset.

## 1. Import Packages & Local Configuration

In [1]:
import sys
import os
import pandas as pd
from pathlib import Path

# Ensure project root is in the path
sys.path.append(str(Path(os.getcwd()).parent))

import config
print("Master Dataset Output:", config.PROCESSED_DIR / "master_dataset.csv")

Master Dataset Output: D:\SLIIT\research_apk\coconut-yield-prediction-system\backend\Coconut_Yield_Prediction\processed\master_dataset.csv


## 2. Load Cleaned Datasets

In [2]:
df_weather = pd.read_csv(config.PROCESSED_DIR / "weather_clean.csv")
df_moisture = pd.read_csv(config.PROCESSED_DIR / "soil_moisture_clean.csv")
df_light_temp = pd.read_csv(config.PROCESSED_DIR / "light_temp_clean.csv")
df_production = pd.read_csv(config.PROCESSED_DIR / "production_clean.csv")

print(f"Weather rows: {len(df_weather)}")
print(f"Moisture rows: {len(df_moisture)}")
print(f"Light/Temp rows: {len(df_light_temp)}")
print(f"Production rows: {len(df_production)}")

Weather rows: 728
Moisture rows: 728
Light/Temp rows: 728
Production rows: 608


## 3. Merge Datasets

We merge Weather, Moisture, Light/Temp, and monthly Production on `District`, `Year`, `Month`.

In [3]:
# Step 1: Merge monthly variables
monthly_merged = pd.merge(
    df_weather, 
    df_moisture, 
    on=["District", "Year", "Month"], 
    how="inner"
)

monthly_features = pd.merge(
    monthly_merged, 
    df_light_temp, 
    on=["District", "Year", "Month"], 
    how="inner"
)
print("Shape of combined monthly weather/moisture variables:", monthly_features.shape)

# Step 2: Merge monthly production records
master_df = pd.merge(
    monthly_features, 
    df_production, 
    on=["District", "Year", "Month"], 
    how="inner"
)
print("Shape after production merge:", master_df.shape)

Shape of combined monthly weather/moisture variables: (728, 13)
Shape after production merge: (608, 15)


## 4. Validate and Export Master Dataset

In [4]:
# Check for missing values
missing_summary = master_df.isnull().sum()
if missing_summary.sum() > 0:
    print("[WARN] Missing values detected in master dataset:")
    print(missing_summary[missing_summary > 0])
else:
    print("[OK] No missing values in master dataset.")

# Chronological sorting
master_df = master_df.sort_values(by=["District", "Year", "Month"]).reset_index(drop=True)

# Export
out_path = config.PROCESSED_DIR / "master_dataset.csv"
master_df.to_csv(out_path, index=False)
print(f"\n[SUCCESS] Master dataset exported successfully to {out_path.name}")
print(master_df.head())

[OK] No missing values in master dataset.

[SUCCESS] Master dataset exported successfully to master_dataset.csv
     District  Year  Month  Rainfall  Temperature  Humidity  \
0  Henwalawwa  2019      1      6.82        25.29     79.06   
1  Henwalawwa  2019      2     79.24        27.23     76.37   
2  Henwalawwa  2019      3     24.18        29.14     64.64   
3  Henwalawwa  2019      4    133.50        29.68     69.60   
4  Henwalawwa  2019      5     30.38        28.97     78.03   

   Soil_Moisture_Top  Soil_Moisture_Root  Soil_Moisture_Diff  Solar_Radiation  \
0               0.60                0.60                0.00            19.71   
1               0.50                0.52                0.02            20.34   
2               0.40                0.45                0.05            24.14   
3               0.42                0.45                0.03            21.75   
4               0.47                0.49                0.02            21.86   

   Temperature_Max  Te